# Synthea Tutorial — synthetic patients → OMOP CDM

This tutorial walks through generating synthetic patient data with
[Synthea](https://github.com/synthetichealth/synthea) and converting it to the
**OMOP Common Data Model** — the standard schema downstream EHR-foundation-model
pipelines (e.g. [`TimelineDataset`](https://github.com/bschilder/TimelineDataset))
expect.

> **Ported from** [`AoU/notebooks/Synthea.ipynb`](https://github.com/bschilder/AoU/blob/main/notebooks/Synthea.ipynb).
> The original notebook used `AoU.phenome.synthea.*`; everything is now in the
> standalone `synthlab` package.

## What you'll do

1. Initialize a `SyntheaRunner` (auto-downloads the Synthea JAR if needed)
2. Configure a small simulation (10 patients) with `SyntheaConfig`
3. Run Synthea → CSV output
4. Convert the CSVs to OMOP CDM 5.4 parquet via `convert_synthea_to_omop`
5. Inspect the resulting OMOP tables

> ⚠️ **Prereq:** Synthea requires Java (>=11). Install via `brew install openjdk@21`
> or your distro's package manager.


In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# Make Java available to subprocess (homebrew openjdk@21 is keg-only by default)
JAVA_BIN = "/opt/homebrew/opt/openjdk@21/bin"
if Path(JAVA_BIN).is_dir() and JAVA_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{JAVA_BIN}:{os.environ['PATH']}"

import subprocess
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True))

openjdk version "21.0.11" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 21.0.11)
OpenJDK 64-Bit Server VM Homebrew (build 21.0.11, mixed mode, sharing)



## 1. Initialize a `SyntheaRunner`

The runner downloads the Synthea JAR (one-time, cached) and exposes a `run()`
method. The first call may take ~30 seconds for the download.


In [2]:
from synthlab import SyntheaRunner, SyntheaConfig, convert_synthea_to_omop

runner = SyntheaRunner()
print(f"Synthea JAR: {runner.jar_path}")


  ░██████╗██╗░░░██╗███╗░░██╗████████╗██╗░░██╗  ██╗░░░░░░█████╗░██████╗░
  ██╔════╝╚██╗░██╔╝████╗░██║╚══██╔══╝██║░░██║  ██║░░░░░██╔══██╗██╔══██╗
  ╚█████╗░░╚████╔╝░██╔██╗██║░░░██║░░░███████║  ██║░░░░░███████║██████╦╝
  ░╚═══██╗░░╚██╔╝░░██║╚████║░░░██║░░░██╔══██║  ██║░░░░░██╔══██║██╔══██╗
  ██████╔╝░░░██║░░░██║░╚███║░░░██║░░░██║░░██║  ███████╗██║░░██║██████╦╝
  ╚═════╝░░░░╚═╝░░░╚═╝░░╚══╝░░░╚═╝░░░╚═╝░░╚═╝  ╚══════╝╚═╝░░╚═╝╚═════╝░
        ▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌│║▌│║▌║▌█║
  ════════════════════════════════════════════════════════════════════
  Synthetic Healthcare Data Toolkit
  ────────────────────────────────────────────────────────────────────
  ◈ EHR        Synthetic patient records (diagnoses, meds, labs)
  ◈ Genomics   Synthetic genotypes with realistic LD structure
  ◈ Imaging    Datasets + synthetic generation (CT, MRI, X-ray)
  ◈ Multimodal Linked EHR + Imaging + Genomics per patient
  ◈ AI Notes   SOAP notes with causal graph analysis
  ══════════════════

## 2. Configure a small simulation

10 patients in Massachusetts with a fixed seed for reproducibility. The output
goes under `./output/tutorial_run/`.


In [3]:
config = SyntheaConfig(
    population_size=10,
    state="Massachusetts",
    seed=42,
    output_dir="output/tutorial_run",
    # Enable CSV exporter (default Synthea output is FHIR JSON);
    # the OMOP converter needs CSV input.
    exporter_flags={
        "exporter.csv.export": "true",
        "exporter.fhir.export": "false",
    },
)
config

SyntheaConfig(population_size=10, seed=42, clinician_seed=None, reference_date=None, gender=None, min_age=0, max_age=140, state='Massachusetts', city=None, config_file=None, modules_dir=None, output_dir='output/tutorial_run', exporter_flags={'exporter.csv.export': 'true', 'exporter.fhir.export': 'false'})

## 3. Run Synthea

This invokes the Synthea JAR via subprocess. Output: CSV files under
`output/tutorial_run/csv/` (patients, encounters, conditions, …).


In [4]:
result = runner.run(config, verbose=False)
print(f"return code: {result['returncode']}")
print(f"output dir:  {result['output_dir']}")

# Inspect the generated CSVs
# Synthea nests its output under `output/csv/` inside the configured output dir.
csv_dir = Path(result["output_dir"]) / "output" / "csv"
if not csv_dir.is_dir():                       # fallback for older layouts
    csv_dir = Path(result["output_dir"]) / "csv"
if csv_dir.is_dir():
    files = sorted(csv_dir.glob("*.csv"))
    print(f"\n{len(files)} CSV files generated:")
    for f in files:
        print(f"  {f.name}: {f.stat().st_size:>10,} bytes")

return code: 0
output dir:  /Users/bschilder/Desktop/synthlab/notebooks/output/tutorial_run

18 CSV files generated:
  allergies.csv:        713 bytes
  careplans.csv:     12,037 bytes
  claims.csv:    814,779 bytes
  claims_transactions.csv:  6,887,148 bytes
  conditions.csv:     78,651 bytes
  devices.csv:     18,137 bytes
  encounters.csv:    341,744 bytes
  imaging_studies.csv:  5,223,972 bytes
  immunizations.csv:     28,000 bytes
  medications.csv:    293,452 bytes
  observations.csv:  2,420,142 bytes
  organizations.csv:      7,175 bytes
  patients.csv:      4,428 bytes
  payer_transitions.csv:     79,709 bytes
  payers.csv:      1,592 bytes
  procedures.csv:    600,793 bytes
  providers.csv:      8,592 bytes
  supplies.csv:     36,729 bytes


## 4. Convert to OMOP CDM 5.4

`convert_synthea_to_omop` reads the Synthea CSVs and writes OMOP-formatted
parquet tables (`person`, `condition_occurrence`, `drug_exposure`,
`measurement`, `visit_occurrence`, etc.).


In [5]:
omop_files = convert_synthea_to_omop(
    synthea_csv_dir=str(csv_dir),
    output_dir=f"{result['output_dir']}/omop",
    cdm_version="5.4",
    output_format="parquet",
    verbose=False,
)
print(f"generated {len(omop_files)} OMOP tables")
for path in sorted(omop_files):
    size = Path(path).stat().st_size if Path(path).exists() else 0
    print(f"  {Path(path).name:<30s} {size:>8,} bytes")

generated 10 OMOP tables
  cdm_source                            0 bytes
  condition_occurrence                  0 bytes
  death                                 0 bytes
  drug_exposure                         0 bytes
  measurement                           0 bytes
  observation                           0 bytes
  observation_period                    0 bytes
  person                                0 bytes
  procedure_occurrence                  0 bytes
  visit_occurrence                      0 bytes


## 5. Inspect the OMOP output

Load a few tables to sanity-check what was produced.


In [6]:
import pandas as pd

omop_dir = Path(result["output_dir"]) / "omop"

for name in ["person", "condition_occurrence", "drug_exposure", "visit_occurrence"]:
    candidates = list(omop_dir.glob(f"{name}*"))
    if not candidates:
        print(f"--- {name}: (not produced) ---\n")
        continue
    path = candidates[0]
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    print(f"--- {name}: {df.shape} ---")
    print(df.head(3))
    print()

--- person: (14, 18) ---
   person_id  gender_concept_id  year_of_birth  month_of_birth  day_of_birth  \
0          1               8507           2009               9            11   
1          2               8507           2017              10            31   
2          3               8532           2007               4            28   

  birth_datetime  race_concept_id  ethnicity_concept_id location_id  \
0     2009-09-11                0                     0         NaN   
1     2017-10-31                0                     0         NaN   
2     2007-04-28                0                     0         NaN   

  provider_id care_site_id                   person_source_value  \
0         NaN          NaN  5e688e99-61b3-5c88-3f60-21df8aaced27   
1         NaN          NaN  76b20010-c318-5754-8c85-983aa538522f   
2         NaN          NaN  46976cf7-b0bf-be20-39a5-9f425a52886d   

  gender_source_value  gender_source_concept_id race_source_value  \
0                   M      

## Next steps

- **Feed into `TimelineDataset`**: the OMOP parquet you just generated drops
  directly into `build_omop_timelines(cfg)` from the
  [TimelineDataset](https://github.com/bschilder/TimelineDataset) repo — see
  its [tutorial](https://github.com/bschilder/TimelineDataset/blob/main/notebooks/tutorial.ipynb).
- **Larger cohorts**: bump `population_size` to 1k+ for realistic train/test
  splits. Synthea runtime scales roughly linearly.
- **MEDS export**: synthlab also ships `meds_etl` integration — see
  `notebooks/Coherent_MultimodalDataset.ipynb`.
